# 04 — The estimator and its parameters

*Chapter 10 · LightGBM · Notebook 4 of 5*

You have built LightGBM's three distinctive pieces by hand: **leaf-wise growth** (Notebook 1), **GOSS**
(Notebook 2), and the **optimal categorical split** (Notebook 3). Now we meet the real estimator —
`LGBMClassifier` / `LGBMRegressor` — and its knobs, each one tied back to the concept that owns it.

The spine of this notebook is the **capacity dial**: `num_leaves` and its floor `min_child_samples`. In
Notebook 1 `num_leaves` was the leaf *budget* of a single tree; here it is the dial you actually tune,
and tuning it **closes the loop** Notebook 1 opened — leaf-wise growth is efficient but grows lopsided
and overfits a single tree, and these two knobs (plus the ensemble) are how that is held in check.

The honest arc is the one from Chapter 08 and Chapter 09: LightGBM's defaults are already sensible, so we
*measure* what each knob does, tune on the training data with cross-validation, and read the test set
**once** at the end.

**Prerequisites.** Notebooks 1–3 of this chapter; Chapter 08 NB 4 (shrinkage: learning rate × number of
trees); Chapter 06/08 (subsampling, feature importance); Chapter 00 (cross-validation, the sealed test).

**What you'll be able to do.**
- Set `num_leaves` and `min_child_samples` knowing what each does to capacity and tree shape.
- Read the learning-rate / n_estimators trade-off, and let early stopping pick the count.
- Recognise which knobs barely move accuracy on clean dense data — and why.
- Tune honestly with `GridSearchCV` and report a single sealed-test number.

## From mechanisms to the estimator

The `LGBMClassifier` and `LGBMRegressor` classes wrap everything you built: they grow trees leaf-wise to
a `num_leaves` budget (NB 1), can sample rows with GOSS (NB 2), and split categorical features with the
sorted-gradient method (NB 3). The same object exposes a few dozen parameters; we group the ones that
matter by the idea behind them, starting with the one that is genuinely new to *tune* here — the capacity
dial — and ending with an honest tuning pass. We work with the classifier; `LGBMRegressor` carries the
identical knobs (only the loss differs).

One note on output. LightGBM prints a short `[Info]` banner every time it fits, and this notebook fits
dozens of models. We expose a single switch, **`LGBM_VERBOSE`**, in the setup cell: at `0` it quiets that
repetitive banner while **still surfacing genuine warnings and errors** (it is not full silencing); set it
to `1` to watch every fit report itself. Let us look at the defaults first.

In [ ]:
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# LightGBM prints a short [Info] banner on every fit, and this notebook fits dozens of
# models. LGBM_VERBOSE quiets that per-fit banner at 0 while still surfacing genuine
# warnings and errors -- it is NOT full silencing. Set it to 1 to see every fit's banner.
LGBM_VERBOSE = 0

X_arr, y = make_classification(
    n_samples=10000, n_features=20, n_informative=8, n_redundant=4,
    n_clusters_per_class=2, class_sep=0.9, flip_y=0.04, random_state=SEED,
)
# pandas-first: a named DataFrame used for both fit and predict avoids scikit-learn's
# spurious "X does not have valid feature names" warning (it fires only on a name mismatch).
X = pd.DataFrame(X_arr, columns=[f"f{i}" for i in range(X_arr.shape[1])])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
print(f"train {Xtr.shape}   test {Xte.shape}   positive rate {ytr.mean():.3f}")

# the resolved defaults we will be moving away from
defaults = LGBMClassifier().get_params()
for k in ["num_leaves", "max_depth", "min_child_samples", "learning_rate",
          "n_estimators", "reg_lambda", "reg_alpha", "subsample", "colsample_bytree"]:
    print(f"  {k:18s} = {defaults[k]}")


## The capacity dial: `num_leaves`

A leaf-wise tree keeps splitting its best leaf until it has `num_leaves` of them (Notebook 1). More
leaves means more capacity — a single tree can carve the data finer. From Chapter 08's bias–variance
picture we expect a single deep tree to **overfit**: its test accuracy should rise, peak, then fall as
`num_leaves` grows. The question this chapter has been building toward: does **boosting** — averaging a
hundred shrunken trees — tame that? We sweep `num_leaves` two ways, as one tree and as a 100-tree
ensemble, and compare.

In [ ]:
def realized_depth(model):
    """Maximum realized depth over the model's trees (from the dumped model)."""
    def depth(node, d=0):
        if "split_index" not in node:
            return d
        return max(depth(node["left_child"], d + 1), depth(node["right_child"], d + 1))
    return max(depth(t["tree_structure"]) for t in model.booster_.dump_model()["tree_info"])


nl_grid = [4, 8, 16, 32, 64, 128]
single_train, single_test, ens_test = [], [], []
print(f"  {'num_leaves':>10s} {'1-tree train':>13s} {'1-tree test':>12s} {'100-tree test':>14s}")
for nl in nl_grid:
    # one tree, allowed to grow (small floor) -- this is the overfit story
    one = LGBMClassifier(num_leaves=nl, n_estimators=1, learning_rate=1.0,
                         min_child_samples=5, random_state=SEED, verbose=LGBM_VERBOSE).fit(Xtr, ytr)
    # the ensemble at the same num_leaves, default floor
    ens = LGBMClassifier(num_leaves=nl, n_estimators=100, learning_rate=0.1,
                         random_state=SEED, verbose=LGBM_VERBOSE).fit(Xtr, ytr)
    single_train.append(accuracy_score(ytr, one.predict(Xtr)))
    single_test.append(accuracy_score(yte, one.predict(Xte)))
    ens_test.append(accuracy_score(yte, ens.predict(Xte)))
    print(f"  {nl:10d} {single_train[-1]:13.4f} {single_test[-1]:12.4f} {ens_test[-1]:14.4f}")


In [ ]:
peak = int(np.argmax(single_test))
fig, ax = plt.subplots(figsize=(7.6, 4.8))
ax.plot(nl_grid, single_test, "o-", color=COLORS["error"], linewidth=2.2,
        label="single tree (test)")
ax.plot(nl_grid, ens_test, "o-", color=COLORS["model"], linewidth=2.2,
        label="100-tree ensemble (test)")
ax.plot(nl_grid[peak], single_test[peak], "o", color=COLORS["highlight"], markersize=15,
        zorder=5, label=f"single-tree peak (num_leaves={nl_grid[peak]})")
ax.set_xscale("log", base=2)
ax.set_xlabel("num_leaves")
ax.set_ylabel("test accuracy")
ax.set_title("the capacity dial: one tree overfits, the ensemble is robust")
ax.legend()
plt.show()


**Read the figure.** The single tree (coral) climbs to a peak near `num_leaves = 64` (the
highlighted dot) and then falls: past that, extra leaves fit training noise and test accuracy drops —
classic overfitting, with its training accuracy climbing toward 0.95 all the while. The 100-tree ensemble
(blue) tells a different story: it rises and then **plateaus** around 0.95, barely moving from 32 leaves
onward (the small dip past 32 is test-set noise, not a trend). **This closes Notebook 1's loop** —
leaf-wise growth really does overfit a single tree, but boosting averages a hundred
such trees and the variance washes out. `num_leaves` is still the capacity dial; the ensemble is
forgiving about where you set it.

## The floor: `min_child_samples`

`num_leaves` caps *how many* leaves; `min_child_samples` (default 20) sets the **minimum number of
training rows a leaf may hold** — the brake on how far the single lopsided tree of Notebook 1 can drill
down. To see it work, we take **one tree**, let it want many leaves (`num_leaves = 256`), and raise the
floor from 1 upward — watching the tree walk from memorizing (tiny leaves) to underfitting (leaves too
big).

In [ ]:
mcs_grid = [1, 5, 20, 50, 100, 300]
mcs_train, mcs_test, mcs_leaves, mcs_depth = [], [], [], []
print(f"  {'min_child_samples':>18s} {'leaves':>7s} {'depth':>6s} {'train':>7s} {'test':>7s}")
for mcs in mcs_grid:
    one = LGBMClassifier(num_leaves=256, n_estimators=1, learning_rate=1.0,
                         min_child_samples=mcs, random_state=SEED,
                         verbose=LGBM_VERBOSE).fit(Xtr, ytr)
    leaves = one.booster_.dump_model()["tree_info"][0]["num_leaves"]
    tr = accuracy_score(ytr, one.predict(Xtr))
    te = accuracy_score(yte, one.predict(Xte))
    mcs_leaves.append(leaves)
    mcs_depth.append(realized_depth(one))
    mcs_train.append(tr)
    mcs_test.append(te)
    print(f"  {mcs:18d} {leaves:7d} {mcs_depth[-1]:6d} {tr:7.4f} {te:7.4f}")


In [ ]:
peak = int(np.argmax(mcs_test))
fig, ax = plt.subplots(figsize=(7.6, 4.8))
ax.plot(mcs_grid, mcs_train, "o-", color=COLORS["muted"], linewidth=2.0, label="train accuracy")
ax.plot(mcs_grid, mcs_test, "o-", color=COLORS["model"], linewidth=2.2, label="test accuracy")
ax.plot(mcs_grid[peak], mcs_test[peak], "o", color=COLORS["highlight"], markersize=15,
        zorder=5, label=f"best test (min_child_samples={mcs_grid[peak]})")
ax.set_xscale("log")
ax.set_xlabel("min_child_samples  (minimum rows per leaf, one tree)")
ax.set_ylabel("accuracy")
ax.set_title("the floor on a single tree: memorize -> sweet spot -> underfit")
ax.legend()
plt.show()


**Read the figure.** Raising the floor walks the single tree across the whole bias–variance range.
At `min_child_samples = 1` it builds all 256 leaves (depth 18) and **memorizes** — train 0.976 but test
only 0.897. Near the default `20` the tree settles to ~174 leaves and test **peaks** (0.902). Push the
floor to 300 and every leaf must hold 300 rows: the tree collapses to 18 leaves (depth 8) and
**underfits** — train and test fall together to 0.858. `min_child_samples` is the brake on the lopsided
single tree of Notebook 1 — the other half of closing its loop. (In a 100-tree ensemble, as the capacity
figure showed, boosting is far more forgiving of the floor; but it stays your guard against leaves that
memorize.)

A rule of thumb ties the two capacity knobs together: keep **`num_leaves < 2^max_depth`**. A balanced
depth-`d` tree has `2^d` leaves; a leaf-wise tree given `num_leaves = 2^d` can instead run one branch down
to depth `2^d − 1` (lopsided). So `num_leaves` is the real capacity knob for LightGBM; `max_depth`
(default `−1`, unlimited) is an optional extra cap, and `min_child_samples` shortens the branches from the
bottom.

## Shrinkage: `learning_rate` × `n_estimators`

This is the dial from Chapter 08 NB 4, unchanged. Each tree's contribution is scaled by the learning rate
`ν`; smaller steps are more careful but need **more** trees to cover the same ground. The two trade off:
a small `ν` with many trees usually reaches a slightly better, flatter optimum than a large `ν` with few.

In [ ]:
configs = [(0.30, 100), (0.10, 100), (0.05, 300), (0.03, 600)]
labels, lr_test = [], []
for lr, ne in configs:
    m = LGBMClassifier(num_leaves=31, n_estimators=ne, learning_rate=lr,
                       random_state=SEED, verbose=LGBM_VERBOSE).fit(Xtr, ytr)
    lr_test.append(accuracy_score(yte, m.predict(Xte)))
    labels.append(f"lr={lr}\nn={ne}")
    print(f"  learning_rate={lr:4.2f}  n_estimators={ne:4d}: test {lr_test[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.bar(range(len(configs)), lr_test, color=COLORS["class_d"],
       edgecolor=COLORS["text"], linewidth=0.5)
ax.set_xticks(range(len(configs)))
ax.set_xticklabels(labels)
ax.set_ylim(0.93, 0.96)
ax.set_ylabel("test accuracy")
ax.set_title("a big rate with few trees trails; smaller rate + more trees recovers")
plt.show()


**Read the figure.** The aggressive `lr=0.3` with only 100 trees trails slightly (0.945); the
default `lr=0.1, n=100` reaches 0.949, and shrinking to `lr=0.05` with 300 trees or `lr=0.03` with 600
trees holds about there (0.949–0.950). The pattern is the familiar one: a smaller learning rate needs more
trees to pay off, and buys a small, flatter gain — at a proportional cost in fit time. You rarely guess
`n_estimators` by hand, though; **early stopping** picks it for you (below).

## The knobs that barely move dense-data accuracy

Three more groups of parameters, grouped by their concept — and an honest expectation for *this* data:

- **`reg_lambda` / `reg_alpha`** — the L2 / L1 penalties on leaf weights from Chapter 09 NB 2. LightGBM
  leaves them **off by default** (`0`), where XGBoost defaults `reg_lambda=1`. That is a difference of
  *posture*, paired with LightGBM's different capacity control (num_leaves + min_child_samples), not
  "less regularized overall."
- **`subsample` / `colsample_bytree`** (a.k.a. `bagging_fraction` / `feature_fraction`) — row and column
  subsampling, the stochasticity of Chapters 06 and 08.
- **`data_sample_strategy='goss'`** — Notebook 2's gradient-based sampling.

We measure them. On clean, dense, moderate data, expect them to be roughly flat.

In [ ]:
print("reg_lambda (num_leaves=64, 200 trees):")
for lam in [0.0, 0.5, 1.0]:
    m = LGBMClassifier(num_leaves=64, n_estimators=200, learning_rate=0.05,
                       reg_lambda=lam, random_state=SEED, verbose=LGBM_VERBOSE).fit(Xtr, ytr)
    print(f"  reg_lambda={lam:4.1f}: test {accuracy_score(yte, m.predict(Xte)):.4f}")

print("\nrow/column/GOSS sampling (num_leaves=64, 200 trees):")
for label, kw in [
    ("gbdt (no sampling)", {}),
    ("GOSS (0.2/0.1)", dict(data_sample_strategy="goss", top_rate=0.2, other_rate=0.1)),
    ("bagging_fraction 0.5", dict(subsample=0.5, subsample_freq=1)),
    ("feature_fraction 0.5", dict(colsample_bytree=0.5)),
]:
    m = LGBMClassifier(num_leaves=64, n_estimators=200, learning_rate=0.05,
                       random_state=SEED, verbose=LGBM_VERBOSE, **kw).fit(Xtr, ytr)
    print(f"  {label:22s}: test {accuracy_score(yte, m.predict(Xte)):.4f}")


**Read the numbers.** `reg_lambda` from 0 to 1 leaves accuracy essentially unchanged (around 0.95):
L2 does **not** lift it here — its job is to shrink leaf weights `−G/(H+λ)` (Chapter 09 NB 2), and turning
it up far enough only underfits. Off-by-default is a posture, not a missing lever, exactly as we found for
XGBoost in Chapter 09 NB 4. The sampling knobs (GOSS, bagging, feature fractions) all land within a few
thousandths of plain `gbdt` — the same lesson as Notebook 2: on dense, moderate data they neither help nor
hurt much; they earn their keep on large, wide, or sparse data, and for speed.

## Early stopping: let a validation set pick `n_estimators`

Rather than guess the number of trees, hold out a slice of the training data and stop adding trees once
its score stops improving. In LightGBM 4.x this is a **callback**: pass
`callbacks=[lgb.early_stopping(rounds)]` and an `eval_set`, and the fitted model exposes `best_iteration_`
— `predict` then uses it automatically, so the test number below is the early-stopped model's, not all
1000 requested trees. (We leave its evaluation log visible — that is the engine reporting where it
stopped.)

In [ ]:
Xtr2, Xval, ytr2, yval = train_test_split(Xtr, ytr, test_size=0.25,
                                          stratify=ytr, random_state=SEED)
es = LGBMClassifier(num_leaves=31, n_estimators=1000, learning_rate=0.05,
                    random_state=SEED, verbose=LGBM_VERBOSE)
es.fit(Xtr2, ytr2, eval_set=[(Xval, yval)], eval_metric="binary_logloss",
       callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
print(f"\nstopped at best_iteration_ = {es.best_iteration_} of 1000 requested")
print(f"sealed test accuracy = {accuracy_score(yte, es.predict(Xte)):.4f}")


## Honest tuning → one sealed test

Chapter 09's XGBoost defaults were aggressive (`eta=0.3`, `depth=6`) and overfit out of the box;
LightGBM's are gentler, so there is less to rescue. Still, the honest procedure is the same: search the
capacity dial on the **training** data with cross-validation, then read the test set **once**. We fix the
number of trees at 100 — early stopping, above, is the principled way to set that — and search
`num_leaves`, `min_child_samples`, and the learning rate. (We fit each candidate single-threaded and
parallelise the search, and leave `GridSearchCV`'s progress visible.)

In [ ]:
default = LGBMClassifier(random_state=SEED, n_jobs=1, verbose=LGBM_VERBOSE).fit(Xtr, ytr)
acc_default = accuracy_score(yte, default.predict(Xte))

grid = {
    "num_leaves": [15, 31, 63],
    "min_child_samples": [10, 20, 50],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100],
}
search = GridSearchCV(LGBMClassifier(random_state=SEED, n_jobs=1, verbose=LGBM_VERBOSE), grid, cv=5,
                      scoring="accuracy", n_jobs=-1, verbose=1)
t0 = time.perf_counter()
search.fit(Xtr, ytr)
elapsed = time.perf_counter() - t0
print(f"\ngrid fit in {elapsed:.1f}s over {len(search.cv_results_['params'])} settings")
acc_tuned = accuracy_score(yte, search.predict(Xte))
print(f"best params: {search.best_params_}")
print(f"best CV score: {search.best_score_:.4f}")
print(f"default sealed test: {acc_default:.4f}")
print(f"tuned   sealed test: {acc_tuned:.4f}   (delta {acc_tuned - acc_default:+.4f})")

fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.bar(["default", "tuned"], [acc_default, acc_tuned],
       color=[COLORS["muted"], COLORS["model"]], edgecolor=COLORS["text"], linewidth=0.5)
ax.set_ylim(0.93, 0.96)
ax.set_ylabel("sealed test accuracy")
ax.set_title(f"tuning bought {acc_tuned - acc_default:+.4f} over the default")
for i, v in enumerate([acc_default, acc_tuned]):
    ax.text(i, v + 0.0010, f"{v:.4f}", ha="center")
plt.show()


**Read the figure.** The tuned model lands at 0.948 against the default's 0.949 — a difference of
about `−0.001`, well inside the noise of a 3000-row test set. Cross-validation preferred a wider tree
(`num_leaves = 63` rather than the default 31, with `min_child_samples = 10`), and that setting won *on
the validation folds* — but on the sealed test it is indistinguishable from the default. That is the
honest outcome on clean data: LightGBM's defaults were already at the ceiling, so the capacity dial mostly
*teaches you about the model* rather than rescuing the score. It bites harder on the messier, larger
problem of the next notebook. Either way, the number you report is this one: chosen on the training data,
the test set read once.

In [ ]:
imp_split = default.booster_.feature_importance(importance_type="split")
imp_gain = default.booster_.feature_importance(importance_type="gain")
print(f"top-5 features by 'split' (how often used) : {np.argsort(-imp_split)[:5].tolist()}")
print(f"top-5 features by 'gain'  (total improvement): {np.argsort(-imp_gain)[:5].tolist()}")
print(f"same most-important feature? {np.argmax(imp_split) == np.argmax(imp_gain)}")


**Read the numbers.** The two importance types **disagree even on the single most important
feature**: `'split'` counts how often a feature is used, `'gain'` sums how much each use improved the
loss, and a feature used in many small splits can top `'split'` while another contributes more total
`'gain'`. Both are computed on the training data and inherit the MDI bias you saw in Chapters 06/08/09.
Treat them as a rough sketch; the honest, held-out reading (permutation importance) is the job of the
capstone.

## Your turn

1. **(easy)** Re-run the capacity-dial sweep (the `num_leaves` figure) on **noisier** data — rebuild
   `make_classification` with `flip_y=0.12` or `class_sep=0.6`. Does the single tree still peak then
   overfit, and does the peak move? Does the 100-tree ensemble stay robust?
2. **(core)** In the `min_child_samples` experiment, add a `max_depth` cap (say `max_depth=6`) alongside
   the floor and re-read the leaf counts and depths. Which knob binds first — the floor or the depth cap —
   and does the `num_leaves < 2^max_depth` rule explain what you see?
3. **(reach)** Replace the `GridSearchCV` fixed `n_estimators=100` with the early-stopping recipe (a
   validation slice + `lgb.early_stopping`): how many trees does it choose, and is the sealed-test score
   meaningfully different from the grid's? Which procedure would you trust more, and why?

## What you built

You learned `LGBMClassifier` / `LGBMRegressor` by tying each knob to a concept you had already built:

- **`num_leaves` + `min_child_samples`** — the leaf-wise capacity dial and its floor. A single tree
  overfits as `num_leaves` grows (peaking near 64 here), the ensemble is robust, and the floor
  `min_child_samples` controls how deep a lopsided branch runs — **closing Notebook 1's loop**.
- **`learning_rate` × `n_estimators`** — Chapter 08's shrinkage trade-off; **early stopping** picks the
  count via a validation set and `lgb.early_stopping`.
- **`reg_lambda` / `reg_alpha`** off by default (a posture, not a lever here), and the **sampling** knobs
  (`subsample`, `colsample_bytree`, GOSS) — all roughly flat on clean dense data.
- **Honest tuning** with `GridSearchCV` on the training data → one sealed test, landing within noise
  (`−0.001`) of an already-strong default.

**Vocabulary.** `num_leaves` · `min_child_samples` · `max_depth=-1` · `num_leaves < 2^max_depth` ·
`learning_rate` × `n_estimators` · `reg_lambda` / `reg_alpha` · `subsample` / `colsample_bytree` ·
`data_sample_strategy='goss'` · `callbacks=[lgb.early_stopping]` · `best_iteration_` · importance
`'split'` vs `'gain'`.

Next: **a demanding case** — a larger problem where tuning, speed, and honest evaluation all bite.

## References

- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* Advances in Neural Information Processing Systems 30 (NeurIPS 2017).
- LightGBM documentation — *Parameters* and *Parameters Tuning* (`num_leaves`, `min_child_samples`,
  `min_data_in_leaf`, the `num_leaves < 2^max_depth` guidance).
- Friedman, J. H. (2001/2002). *Greedy Function Approximation* (DOI
  [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451)) and *Stochastic Gradient Boosting*
  (DOI [10.1016/S0167-9473(01)00065-2](https://doi.org/10.1016/S0167-9473(01)00065-2)). (Shrinkage and
  subsampling.)

*Previous: 03 — The optimal categorical split.*  ·  *Next: 05 — A demanding case.*